In [15]:
from sklearn.datasets import load_iris, load_diabetes, load_wine
from sklearn.svm import SVC, SVR
from sklearn.linear_model import SGDClassifier, LinearRegression
from sklearn.metrics import f1_score, accuracy_score, root_mean_squared_error, mean_squared_error
from sklearn.model_selection import KFold, StratifiedKFold
import numpy as np

class_metrics = [
    {
        "name": "Acc",
        "func": accuracy_score
    },
    {
        "name": "F1score",
        "func": f1_score
    },
]

regs_metrics = [
    {
        "name": "MSE",
        "func": mean_squared_error
    },
    {
        "name": "RMSE",
        "func": root_mean_squared_error
    },
]

class_models = [
    {
        "name": "SVC",
        "class": SVC,
        "metrics": class_metrics
    },
    {
        "name": "SGDClassifier",
        "class": SGDClassifier,
        "metrics": class_metrics
    }
]

regs_models = [
    {
        "name": "SVR",
        "class": SVR,
        "metrics": regs_metrics
    },
    {
        "name": "SGDClassifier",
        "class": LinearRegression,
        "metrics": regs_metrics
    }
]

exec_params =[
        {
            "name": "Iris",
            "load": load_iris,
            "models": class_models
        },
        {
            "name": "Wine",
            "load": load_wine,
            "models": class_models
        },
        {
            "name": "Diabetes",
            "load": load_diabetes,
            "models": regs_models
        }
    ]




for exec in exec_params:
    print(f"Processando dataset: {exec["name"]}")
    dataset = exec["load"]()
    x = dataset.data
    y = dataset.target

    for model in exec["models"]:
        print(f"Avaliando modelo - {model["name"]}")
        model_obj = model["class"]()
        metrics = []
        for metric in model["metrics"]:
            metrics.append([])

        kf = StratifiedKFold(n_splits=5, shuffle=True)
        for i, (train_index, test_index) in enumerate(kf.split(x, y)):
            model_obj.fit(x[train_index], y[train_index])
            result = model_obj.predict(x[test_index])

            for idx, metric in enumerate(model["metrics"]):
                if metric["name"] == "F1score":
                    metrics[idx].append(metric["func"](y[test_index], result, average="macro"))
                else:
                    metrics[idx].append(metric["func"](y[test_index], result))

        for metric in model["metrics"]:
            print(f"{metric["name"]}: {np.mean(metrics[idx])}")

        print("\n")


Processando dataset: Iris
Avaliando modelo - SVC
Acc: 0.973165236323131
F1score: 0.973165236323131


Avaliando modelo - SGDClassifier
Acc: 0.8415920136757397
F1score: 0.8415920136757397


Processando dataset: Wine
Avaliando modelo - SVC
Acc: 0.626062718110773
F1score: 0.626062718110773


Avaliando modelo - SGDClassifier
Acc: 0.5836532535896269
F1score: 0.5836532535896269


Processando dataset: Diabetes
Avaliando modelo - SVR
MSE: 70.45720362231702
RMSE: 70.45720362231702


Avaliando modelo - SGDClassifier
MSE: 54.54177719135449
RMSE: 54.54177719135449




c:\Users\marco\OneDrive\Documents\CS\Intern\Bootcamp TCS\AI_exercises\myenv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\marco\OneDrive\Documents\CS\Intern\Bootcamp TCS\AI_exercises\myenv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [16]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import seaborn as sns

In [17]:
dataset_wine = load_wine()

In [ ]:
x_wine = dataset_wine.data
x_wine_cols = dataset_wine.feature_names

y_wine = dataset_wine.target
y_wine_cols = dataset_wine.target_names

In [ ]:
df = pd.DataFrame(data = x_wine, columns = x_wine_cols)
df["class"] = y_wine

In [ ]:
df[x_wine_cols].describe().round(2)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
count,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00
mean,13.00,2.34,2.37,19.49,99.74,2.30,2.03,0.36,1.59,5.06,0.96,2.61,746.89
std,0.81,1.12,0.27,3.34,14.28,0.63,1.00,0.12,0.57,2.32,0.23,0.71,314.91
min,11.03,0.74,1.36,10.60,70.00,0.98,0.34,0.13,0.41,1.28,0.48,1.27,278.00
25%,12.36,1.60,2.21,17.20,88.00,1.74,1.20,0.27,1.25,3.22,0.78,1.94,500.50
50%,13.05,1.87,2.36,19.50,98.00,2.36,2.13,0.34,1.56,4.69,0.96,2.78,673.50
75%,13.68,3.08,2.56,21.50,107.00,2.80,2.88,0.44,1.95,6.20,1.12,3.17,985.00
max,14.83,5.80,3.23,30.00,162.00,3.88,5.08,0.66,3.58,13.00,1.71,4.00,1680.00


In [22]:
modelo_sgd = SGDClassifier(random_state=42)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []

for train_index, test_index in kf.split(x_wine, y_wine):
    # 1. Separar os dados PRIMEIRO (usando o array original sem escala)
    X_train, X_test = x_wine[train_index], x_wine[test_index]
    y_train, y_test = y_wine[train_index], y_wine[test_index]
    
    # 2. Instanciar um novo scaler a cada fold (opcional, mas garante isolamento)
    scaler = StandardScaler()
    
    # 3. FIT e TRANSFORM apenas no conjunto de treino
    X_train_scaled = scaler.fit_transform(X_train)
    
    # 4. APENAS TRANSFORM no conjunto de teste (usando o padrão aprendido no treino)
    X_test_scaled = scaler.transform(X_test)
    
    # Treino e Predição com os dados corretamente escalonados
    modelo_sgd.fit(X_train_scaled, y_train)
    pred = modelo_sgd.predict(X_test_scaled)
    
    # Armazenando métricas
    acc_scores.append(accuracy_score(y_test, pred))
    f1_scores.append(f1_score(y_test, pred, average="macro"))

print(f"Acurácia sem vazamento de dados: {np.mean(acc_scores):.4f}")
print(f"F1-score sem vazamento de dados: {np.mean(f1_scores):.4f}")

Acurácia sem vazamento de dados: 0.9833
F1-score sem vazamento de dados: 0.9825


o modelo foi de 58% para 98%